# W6D3 — Masked Autoencoders: How Much Can You Delete? — Guided

**Week 6 · Day 3 · Representation Learning** · Lab

Yesterday you added noise to every pixel and the reconstructions got better for it. Today you
delete **three quarters of the image** and ask a model to put it back.

The warm-up is you doing the pretext task by hand: your four toy patches from Monday, three of them
hidden, and you writing down your guess before you see the answer. The gap between your guess and
the truth is the loss — felt before it is defined.

Then the real thing. A pretrained MAE, and the finding that a hyperparameter is the lesson: you
sweep the masking ratio from 0.25 to 0.90, watch the reconstruction MSE climb, and find the point
where the content genuinely stops coming back. You also verify the efficiency claim from the
lecture by counting tokens — at 75% masking the encoder processes **49 patches, not 196** — and by
timing it, which is a different and better kind of evidence.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٦ · اليوم ٣ — المُرمِّزات المُقنَّعة: كم يمكنك أن تحذف؟

**الأسبوع السادس · اليوم الثالث · تعلّم التمثيل** · معمل

أضفت أمس ضوضاء إلى كل بكسل فتحسّنت إعادة البناء بها. واليوم تحذف **ثلاثة أرباع الصورة** وتطلب من
نموذج أن يُعيدها.

والإحماء هو أنت تؤدّي المهمة الذريعة بيدك: رقعك التجريبية الأربع من الاثنين، ثلاثٌ منها مُخفاة، وأنت
تُدوّن تخمينك قبل أن ترى الجواب. والفجوة بين تخمينك والحقيقة هي الخسارة — محسوسةً قبل أن تُعرَّف.

ثم الشيء الحقيقي. مُرمِّز مُقنَّع مُدرَّب مسبقًا، والنتيجة أن معاملًا فائقًا هو الدرس: تمسح نسبة الإخفاء
من ٠٫٢٥ إلى ٠٫٩٠، وتراقب خطأ إعادة البناء يتسلّق، وتجد النقطة التي يكفّ عندها المحتوى عن العودة
فعلًا. وتتحقّق أيضًا من ادّعاء الكفاءة في المحاضرة بعدّ الرموز — فعند إخفاء ٧٥٪ يعالج المُرمِّز
**٤٩ رقعة لا ١٩٦** — وبقياس الزمن، وهو دليل من نوع آخر أفضل.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Split a set of patches into a visible set and a masked set, and assert the split is a partition.
- Drive a pretrained `ViTMAE` at any masking ratio, including a masking pattern you choose yourself.
- Count the tokens the encoder actually processes, and say why that number and not 196 is what the
  efficiency claim is about.
- Compute reconstruction error on the masked patches only, and say why including the visible ones
  would make the number meaningless.
- Read a sweep and name the ratio at which the content stops being recoverable.
- Turn an MAE into a feature extractor by throwing away the half of it you spent all the compute on.
- Explain why random masking is an easier pretext task than block masking, and what that says about
  what the model is doing.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقسم مجموعة رقع إلى مجموعة ظاهرة وأخرى مُخفاة، وأن تفحص أن القسمة تجزئة تامّة.
- أن تُشغّل `ViTMAE` مُدرَّبًا مسبقًا عند أي نسبة إخفاء، بما فيها نمط إخفاء تختاره أنت.
- أن تعدّ الرموز التي يعالجها المُرمِّز فعلًا، وأن تقول لماذا هذا العدد لا ١٩٦ هو موضوع ادّعاء الكفاءة.
- أن تحسب خطأ إعادة البناء على الرقع المُخفاة وحدها، وأن تقول لماذا يجعل إدخال الظاهرة الرقمَ بلا
  معنى.
- أن تقرأ مسحًا وتُسمّي النسبة التي يكفّ عندها المحتوى عن كونه قابلًا للاسترجاع.
- أن تحوّل مُرمِّزًا مُقنَّعًا إلى مُستخرِج تمثيلات برمي النصف الذي أنفقتَ عليه كل الحساب.
- أن تشرح لماذا الإخفاء العشوائي مهمّة ذريعة أسهل من إخفاء الكتل، وماذا يقول ذلك عمّا يفعله النموذج.

</div>

## About the data

`small_image_5class` for the third day running — the same 400 images, 224×224, because tomorrow's
comparison depends on it.

Also loaded: **`patch_check.json`, your artefact from Monday.** The warm-up reads the four toy patch
vectors straight out of it. If you were not here on Monday, `load_artefact` falls back to the
reference copy in `solutions_cache` and prints a line saying so — nothing is blocked.

**Nothing is trained today.** The MAE arrives pretrained on ImageNet-1K, which is the only reason
any of this fits in an afternoon: MAE pretraining is measured in hundreds of GPU-hours.

**First run downloads** `facebook/vit-mae-base` — 449 MB, cached afterwards. Every model call in the
lab is capped at **8 images**, which is the number in `SWEEP_IMAGES` in the setup cell. One forward
pass over those eight is about **0.1 s** at 75% masking, the four-ratio sweep is a few seconds, and
the slowest thing in the notebook is the first download.

**One thing to know about the reconstructions before you see them.** MAE's decoder is deliberately
small and it was trained on ImageNet, not on 400 COCO crops. The reconstructions of masked regions
are coarse and blurry — recognisably the right *kind* of thing in the right place, not a photograph.
That is the expected result and not a sign that you loaded it wrong.

<div dir="rtl" align="right">

## عن البيانات

`small_image_5class` لليوم الثالث على التوالي — الصور الأربعمئة نفسها، ٢٢٤×٢٢٤، لأن مقارنة الغد
تعتمد عليها.

ويُحمَّل أيضًا: **`patch_check.json`، أثرك من يوم الاثنين.** فالإحماء يقرأ متّجهات الرقع التجريبية
الأربعة منه مباشرةً. وإن لم تكن حاضرًا الاثنين رجع `load_artefact` إلى النسخة المرجعية في
`solutions_cache` وطبع سطرًا يقول ذلك — فلا شيء يتعطّل.

**ولا يُدرَّب شيء اليوم.** فالمُرمِّز المُقنَّع يأتي مُدرَّبًا مسبقًا على ImageNet-1K، وهذا وحده سبب
اتّساع الأمر لظهيرة واحدة: فتدريب المُرمِّزات المُقنَّعة يُقاس بمئات ساعات وحدات المعالجة الرسومية.

**التشغيل الأول ينزّل** النموذج `facebook/vit-mae-base` — ‏٤٤٩ ميجابايت، ثم يُخزَّن. وكل نداء نموذج
في المعمل مُقيَّد بـ**ثماني صور**، وهو العدد في `SWEEP_IMAGES` في خلية الإعداد. والتمريرة الواحدة على
الثماني نحو **٠٫١ ثانية** عند إخفاء ٧٥٪، ومسح النسب الأربع بضع ثوانٍ، وأبطأ ما في الدفتر هو التنزيل
الأول.

**وشيء ينبغي أن تعرفه عن إعادة البناء قبل أن تراها.** مُفكِّك المُرمِّز المُقنَّع صغير عن قصد، وقد
دُرِّب على ImageNet لا على أربعمئة قصاصة من COCO. فإعادة بناء المناطق المُخفاة خشنة وضبابية — من
**نوع** الشيء الصحيح في الموضع الصحيح، لا صورة فوتوغرافية. وهذه هي النتيجة المتوقّعة لا علامة على
سوء التحميل.

</div>

## Setup

`ViTMAEForPreTraining` is the whole model: encoder, decoder and the loss. `ViTMAEModel` — reachable
as `.vit` on it — is the encoder alone, and by Friday that is the only part anyone keeps.

<div dir="rtl" align="right">

## الإعداد

`ViTMAEForPreTraining` هو النموذج كاملًا: المُرمِّز والمُفكِّك والخسارة. و`ViTMAEModel` — ويُوصل إليه
بـ`.vit` — هو المُرمِّز وحده، وهو بحلول الجمعة الجزء الوحيد الذي يحتفظ به أحد.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("transformers", "torch", "matplotlib", "pandas", "pyarrow")
seed_everything(42)

import json
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

use_course_style()
np.set_printoptions(precision=4, suppress=True)
torch.set_grad_enabled(False)          # nothing is trained today

SEED = 42
CHECKPOINT = "facebook/vit-mae-base"
IMAGE_SIZE = 224
PATCH_SIZE = 16
GRID = IMAGE_SIZE // PATCH_SIZE        # 14
N_PATCHES = GRID * GRID                # 196
SWEEP_IMAGES = 8                       # every model call in this lab is capped here
RATIOS = (0.25, 0.50, 0.75, 0.90)

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
PATHS = sorted(IMAGE_ROOT.rglob("*.jpg"))
LABELS = np.array([p.parent.name for p in PATHS])

# Monday's artefact — the four toy patches the warm-up masks by hand.
PATCH_CHECK = json.loads(load_artefact("patch_check.json").read_text(encoding="utf-8"))
TOY_PATCHES = np.array(PATCH_CHECK["toy_patches"])
TOY_IMAGE = np.array(PATCH_CHECK["toy_image"])

RECON_DIR = ARTEFACT_DIR / "mae_recon"
RECON_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(PATHS)} images | toy patches from Monday: {TOY_PATCHES.tolist()}")
print(f"Monday recorded sequence length {PATCH_CHECK['sequence_length']} "
      f"for {PATCH_CHECK['n_patches']} patches")
print(versions(), "| device:", device())

## Section 1 — Warm-up: be the model  (≈25 min)

Everything here works. Monday's four patches are back:

```
patch 1  [1, 2, 0, 3]      patch 2  [0, 1, 1, 0]
patch 3  [2, 1, 1, 0]      patch 4  [4, 2, 2, 3]
```

The first cell hides three of the four — that is 75% masking, the real MAE's setting — and shows
you patch 1 and nothing else.

**Then stop and write your guess in the markdown cell.** Four numbers for each of the three hidden
patches, twelve numbers in total. Guess honestly; you have exactly the information the encoder has.
Then run the second cell, which reveals the truth and computes the mean squared error between your
guess and it. That number is the loss. Every masked autoencoder ever trained is a program that
does what you just did, several hundred million times.

Then the same thing at 25% masking, where three of the four patches are visible. It is obviously
an easier *task* — and the toy's loss goes **up**, because with only four patches the number is
decided by which single patch you happened to hide, and patch 4 is the extreme one. That is worth
seeing: a four-patch example cannot support a claim about averages. Task 2.4 makes the same claim
on 196 patches, where it holds.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: كن أنت النموذج (نحو ٢٥ دقيقة)

كل ما هنا يعمل. ورقع الاثنين الأربع عادت:

```
الرقعة ١  [1, 2, 0, 3]      الرقعة ٢  [0, 1, 1, 0]
الرقعة ٣  [2, 1, 1, 0]      الرقعة ٤  [4, 2, 2, 3]
```

تُخفي الخلية الأولى ثلاثًا من الأربع — أي إخفاء ٧٥٪، وهو إعداد المُرمِّز المُقنَّع الحقيقي — وتعرض عليك
الرقعة الأولى ولا شيء غيرها.

**ثم قف واكتب تخمينك في خلية Markdown.** أربعة أعداد لكل رقعة من الثلاث المُخفاة، اثنا عشر عددًا
جملةً. وخمّن بصدق؛ فعندك بالضبط ما عند المُرمِّز من معلومات. ثم شغّل الخلية الثانية، فتكشف الحقيقة
وتحسب الخطأ التربيعي المتوسّط بين تخمينك وبينها. وذلك العدد هو الخسارة. وكل مُرمِّز مُقنَّع دُرِّب يومًا
هو برنامج يفعل ما فعلتَه للتوّ، بضع مئات من ملايين المرّات.

ثم الشيء نفسه عند إخفاء ٢٥٪، حيث تظهر ثلاث رقع من الأربع. والمهمّة أسهل بجلاء — وترتفع خسارة المثال
التجريبي مع ذلك، لأن الرقم مع أربع رقع فقط تحدّده الرقعة الواحدة التي صادف أن أخفيتها، والرقعة الرابعة
هي المتطرّفة. ويستحقّ هذا أن يُرى: فمثالٌ بأربع رقع لا يحمل ادّعاءً عن المتوسّطات. وتسوق المهمة ٢٫٤
الادّعاء نفسه على ١٩٦ رقعة حيث يصحّ.

</div>

In [ ]:
VISIBLE = [0]                 # patch 1 is the only one you get to see
MASKED = [1, 2, 3]            # 3 of 4 hidden = 75%, the real MAE's setting

shown = np.full_like(TOY_PATCHES, -1)
shown[VISIBLE] = TOY_PATCHES[VISIBLE]

print("the 4x4 image, with 75% of its patches deleted:")
for i, patch in enumerate(shown, start=1):
    print(f"  patch {i}: {'?' * 4 if i - 1 in MASKED else patch.tolist()}")
print(f"\nvisible {len(VISIBLE)} patch, masked {len(MASKED)} patches "
      f"= {len(MASKED) / len(TOY_PATCHES):.0%} masked")
print("\nNow write your twelve numbers in the next cell before you run anything else.")

**Your guess.** Replace these three lines with your numbers, then run the next cell.

```
patch 2: [ ?, ?, ?, ? ]
patch 3: [ ?, ?, ?, ? ]
patch 4: [ ?, ?, ?, ? ]
```

<div dir="rtl" align="right">

**تخمينك.** استبدل هذه الأسطر الثلاثة بأعدادك، ثم شغّل الخلية التالية.

```
الرقعة ٢: [ ?, ?, ?, ? ]
الرقعة ٣: [ ?, ?, ?, ? ]
الرقعة ٤: [ ?, ?, ?, ? ]
```

</div>

In [ ]:
# Put your guess here — three patches, four numbers each. The default below is the laziest
# possible guess: the mean of the one patch you were shown, repeated. Beat it.
GUESS = np.full((3, 4), TOY_PATCHES[VISIBLE].mean())

truth = TOY_PATCHES[MASKED]
YOUR_LOSS = float(((GUESS - truth) ** 2).mean())
BASELINE_LOSS = float(((np.full_like(truth, float(TOY_PATCHES[VISIBLE].mean())) - truth) ** 2).mean())

print(f"your guess:\n{GUESS}")
print(f"the truth :\n{truth}")
print(f"\nmean squared error between them: {YOUR_LOSS:.3f}")
print(f"the lazy baseline (predict the visible patch's mean everywhere): {BASELINE_LOSS:.3f}")
print("\nThat number is the loss. An MAE minimises exactly it, over masked patches only.")

# Now the same task at 25% masking: three patches visible, one hidden.
easy_visible, easy_masked = [0, 1, 2], [3]
easy_baseline = np.full((1, 4), TOY_PATCHES[easy_visible].mean())
EASY_LOSS = float(((easy_baseline - TOY_PATCHES[easy_masked]) ** 2).mean())

print(f"\nat 25% masking the same lazy guess scores {EASY_LOSS:.3f} against "
      f"{BASELINE_LOSS:.3f} at 75%")
print("Higher — on four patches, which single patch you hide decides the number, and patch 4")
print("is the extreme one. The claim 'more context, lower loss' is about an average over many")
print("patches, and task 2.4 measures it properly on 196.")

## Section 2 — Core: six tasks  (≈60 min)

1. `mask_patches(n_patches, ratio, seed)` — the partition, asserted.
2. Load the MAE and count the tokens the encoder actually sees.
3. The triple: original, masked input, reconstruction.
4. The ratio sweep: 0.25, 0.50, 0.75, 0.90.
5. Time it, and compare the measurement against the lecture's arithmetic.
6. Throw the decoder away.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. `mask_patches(n_patches, ratio, seed)` — القسمة، مفحوصةً.
٢. حمّل المُرمِّز المُقنَّع وعُدّ الرموز التي يراها المُرمِّز فعلًا.
٣. الثلاثية: الأصل، والدخل المُخفى، وإعادة البناء.
٤. مسح النسب: ٠٫٢٥ و٠٫٥٠ و٠٫٧٥ و٠٫٩٠.
٥. قِس الزمن، وقارن القياس بحساب المحاضرة.
٦. ارمِ المُفكِّك.

</div>

### Task 2.1 — the mask, as a partition

Write `mask_patches(n_patches, ratio, seed)` returning two integer arrays: the visible indices and
the masked indices.

Three properties make it a mask rather than a bug, and all three are checkable:

1. `len(masked) == round(n_patches * ratio)`.
2. The two sets are **disjoint** — no index appears in both.
3. Together they **cover everything** — their union is `0 … n_patches - 1`.

Miss the third and you silently drop patches: the model sees fewer tokens than you think, the loss
is computed over a smaller set than you think, and every number downstream is wrong in a way no
error message will mention. Assert all three now.

<div dir="rtl" align="right">

### المهمة ٢٫١ — القناع تجزئةً تامّة

اكتب `mask_patches(n_patches, ratio, seed)` تُعيد مصفوفتَي أعداد صحيحة: الفهارس الظاهرة والفهارس
المُخفاة.

وثلاث خصائص تجعله قناعًا لا خللًا، وكلها قابلة للفحص:

١. `len(masked) == round(n_patches * ratio)`.
٢. المجموعتان **منفصلتان** — لا فهرس في كلتيهما.
٣. وتُغطّيان معًا **كل شيء** — فاتّحادهما هو `0 … n_patches - 1`.

وإن أخطأت الثالثة أسقطتَ رقعًا صامتًا: يرى النموذج رموزًا أقلّ ممّا تظنّ، وتُحسب الخسارة على مجموعة
أصغر ممّا تظنّ، ويصير كل رقم لاحق خاطئًا على نحوٍ لن تذكره أيّ رسالة خطأ. افحص الثلاث الآن.

</div>

In [ ]:
def mask_patches(n_patches, ratio, seed=SEED):
    """Split patch indices into (visible, masked). A partition, not a selection."""
    # TODO: Permute the indices with a seeded generator, take round(n_patches * ratio) of them as masked and the rest as visible, and return both sorted.
    # مهمة: بدّل الفهارس بمولّد مُبذَّر، وخذ `round(n_patches * ratio)` منها مُخفاةً والباقي ظاهرًا، وأعِد كليهما مرتّبين.


# TODO: Check mask_patches at every ratio in RATIOS: the masked count, disjointness, and that the union covers every index.
# مهمة: افحص `mask_patches` عند كل نسبة في `RATIOS`: عدد المُخفى، والانفصال، وأن الاتّحاد يُغطّي كل فهرس.

### Task 2.2 — load it, and count the tokens

`ViTMAEForPreTraining` holds three things: the encoder (`.vit`), a small decoder, and the loss.

Its masking is random and it is driven by an argument called `noise`: the model sorts the 196
patches by that vector and keeps the lowest-scoring `1 − ratio` of them. Pass your own `noise` and
you control the mask exactly — which is how the sweep stays reproducible and how the stretch
section builds a block mask instead of a random one.

Run one batch at `mask_ratio=0.75` and print the encoder's output shape. **It is 50 tokens, not
196** — and the 50 is 49 visible patches plus the `[CLS]` you met on Monday. Print both numbers and
say which is which, because "49" and "50" both appear in the MAE paper and conflating them is how
people end up reporting the wrong sequence length.

That 49 is the whole efficiency argument: the encoder — the expensive part, twelve blocks of it —
never sees the other 147 patches at all. They are not masked-out zeros being processed anyway.
They are absent.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — حمّله، وعُدّ الرموز

يحمل `ViTMAEForPreTraining` ثلاثة أشياء: المُرمِّز (`.vit`)، ومُفكِّكًا صغيرًا، والخسارة.

وإخفاؤه عشوائي يقوده وسيط اسمه `noise`: يرتّب النموذج الرقع الـ١٩٦ بهذا المتّجه ويُبقي الأدنى منها
بنسبة `1 − ratio`. ومرّر `noise` خاصًّا بك تتحكّم في القناع تمامًا — وهكذا يبقى المسح قابلًا لإعادة
الإنتاج، وهكذا يبني قسم التوسّع قناع كتلة بدل قناع عشوائي.

شغّل دفعةً واحدة عند `mask_ratio=0.75` واطبع شكل خرج المُرمِّز. **إنه ٥٠ رمزًا لا ١٩٦** — والخمسون هي
٤٩ رقعة ظاهرة زائد `[CLS]` الذي قابلته الاثنين. اطبع العددين وقل أيّهما أيّ، فـ«٤٩» و«٥٠» كلاهما
يظهر في ورقة المُرمِّز المُقنَّع، والخلط بينهما هو ما يجعل الناس يُبلّغون بطول متتالية خاطئ.

وهذه الـ٤٩ هي حجّة الكفاءة كلها: فالمُرمِّز — الجزء المكلف، اثنتا عشرة كتلة منه — لا يرى الرقع الـ١٤٧
الأخرى إطلاقًا. فهي ليست أصفارًا مُقنَّعة تُعالَج على أي حال. هي غائبة.

</div>

In [ ]:
from transformers import AutoImageProcessor, ViTMAEForPreTraining

# TODO: Load the model and processor, set the mask ratio to 0.75, run the encoder alone on SWEEP_IMAGES images with a seeded noise vector, and print the token counts.
# مهمة: حمّل النموذج والمعالج، واضبط نسبة الإخفاء على ٠٫٧٥، وشغّل المُرمِّز وحده على `SWEEP_IMAGES` صورة بمتّجه ضوضاء مُبذَّر، واطبع أعداد الرموز.
def set_ratio(ratio):
    """The ratio lives on the config, and the embeddings hold their own reference to it."""
    mae.config.mask_ratio = ratio
    mae.vit.embeddings.config.mask_ratio = ratio
def noise_for(ratio, seed=SEED):
    """A reproducible noise vector — the model keeps the lowest-scoring patches."""
    generator = torch.Generator().manual_seed(seed)
    return torch.rand(len(PIXELS), N_PATCHES, generator=generator)

### Task 2.3 — the triple

For four images, display three panels each: the **original**, the **masked input** — grey squares
where the model saw nothing — and the **reconstruction**.

Two mechanical things you need.

`mae.patchify(pixels)` turns a batch of images into `(batch, 196, 768)` patch vectors, and
`mae.unpatchify(patches)` turns them back into images. The model's `logits` are already in patch
form, so `unpatchify(logits)` is the reconstruction. Note that the model reconstructs **every**
patch including the visible ones; the standard picture pastes the true visible patches back over
its output, so that what you are looking at is only its guesses. Do that.

`outputs.mask` is `(batch, 196)`, with 1 where a patch was masked. It is also how you build the
grey-square panel.

The reconstructions are coarse. The shape and the colour of the object are usually right, the
texture is not, and that is what a small decoder trained on a pixel MSE produces.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الثلاثية

لأربع صور، اعرض ثلاث لوحات لكلٍّ: **الأصل**، و**الدخل المُخفى** — مربّعات رمادية حيث لم يرَ النموذج
شيئًا — و**إعادة البناء**.

وتحتاج أمرين تقنيَّين.

`mae.patchify(pixels)` يحوّل دفعة صور إلى متّجهات رقع بالشكل `(batch, 196, 768)`، و
`mae.unpatchify(patches)` يُعيدها صورًا. ومخرجات النموذج `logits` بصيغة الرقع أصلًا، فـ
`unpatchify(logits)` هي إعادة البناء. ولاحظ أن النموذج يُعيد بناء **كل** رقعة بما فيها الظاهرة؛
والصورة المعتادة تُعيد لصق الرقع الظاهرة الحقيقية فوق خرجه، فلا يكون ما تنظر إليه إلا تخميناته.
افعل ذلك.

و`outputs.mask` بالشكل `(batch, 196)`، وفيه ١ حيث أُخفيت الرقعة. وهو أيضًا ما تبني به لوحة المربّعات
الرمادية.

وإعادة البناء خشنة. فشكل الجسم ولونه صحيحان غالبًا، والنسيج لا، وهذا ما يُنتجه مُفكِّك صغير دُرِّب على
خطأ تربيعي على البكسلات.

</div>

In [ ]:
MEAN = torch.tensor(processor.image_mean).view(1, 3, 1, 1)
STD = torch.tensor(processor.image_std).view(1, 3, 1, 1)


def to_display(batch):
    """Undo the processor's normalisation and return (n, H, W, 3) in [0, 1]."""
    return (batch * STD + MEAN).clamp(0, 1).permute(0, 2, 3, 1).numpy()


# TODO: Write reconstruct(noise) returning the model output plus the masked-input and blended images, then display original / masked / reconstruction for four images at ratio 0.75.
# مهمة: اكتب `reconstruct(noise)` تُعيد خرج النموذج مع صورتَي الدخل المُخفى والممزوج، ثم اعرض الأصل والمُخفى وإعادة البناء لأربع صور عند النسبة ٠٫٧٥.

### Task 2.4 — the ratio sweep

Run the same eight images at 0.25, 0.50, 0.75 and 0.90, and record the reconstruction MSE **on the
masked patches only**.

Why only the masked ones: the model is handed the visible patches, and copying them into its output
is free. Include them in the average and you are measuring mostly how well the model can copy,
which is a number that goes *down* as you mask less for a reason that has nothing to do with what
the model understood. The masked patches are the only place a prediction was actually required.

Compute it yourself from `logits`, `mask` and `patchify(pixels)`, then compare against
`outputs.loss` — they should agree to floating-point precision, and if they do not, your mask is
the wrong way round.

Save the four image grids. Then look at them and answer the question the sweep exists for: **at
what ratio does the content stop coming back?** Not "get blurrier" — stop being recoverable.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — مسح النسب

شغّل الصور الثماني نفسها عند ٠٫٢٥ و٠٫٥٠ و٠٫٧٥ و٠٫٩٠، وسجّل خطأ إعادة البناء **على الرقع المُخفاة
وحدها**.

ولماذا المُخفاة وحدها: لأن الرقع الظاهرة تُعطى للنموذج، ونسخها إلى خرجه مجّاني. وأدخِلها في المتوسّط
تكن تقيس أساسًا قدرته على النسخ، وهو رقم يهبط كلّما قلّ الإخفاء لسبب لا علاقة له بما فهمه النموذج.
فالرقع المُخفاة هي الموضع الوحيد الذي لزم فيه تنبّؤ فعلًا.

احسبه بنفسك من `logits` و`mask` و`patchify(pixels)`، ثم قارنه بـ`outputs.loss` — ويجب أن يتّفقا إلى
دقّة الفاصلة العائمة، فإن لم يتّفقا فقناعك مقلوب.

احفظ شبكات الصور الأربع. ثم انظر إليها وأجب عن السؤال الذي وُجد المسح لأجله: **عند أي نسبة يكفّ
المحتوى عن العودة؟** لا «يزداد ضبابية» — بل يكفّ عن كونه قابلًا للاسترجاع.

</div>

In [ ]:
def masked_mse(outputs):
    """Mean squared error over the masked patches only. Compare with outputs.loss."""
    truth = mae.patchify(PIXELS)
    per_patch = ((outputs.logits - truth) ** 2).mean(dim=-1)            # (batch, 196)
    return float((per_patch * outputs.mask).sum() / outputs.mask.sum())


# TODO: Sweep the four ratios, recording visible token count and masked-patch MSE for each, saving one image grid per ratio, and checking your MSE against the model's own loss.
# مهمة: امسح النسب الأربع مسجّلًا عدد الرموز الظاهرة وخطأ الرقع المُخفاة لكلٍّ، واحفظ شبكة صور لكل نسبة، وافحص خطأك مقابل خسارة النموذج نفسها.

### Task 2.5 — the arithmetic, measured

The lecture's claim was that masking makes the encoder cheaper *quadratically*, because attention
cost goes with the square of the sequence length: `49²` = 2,401 against `196²` = 38,416, exactly 16
times fewer scores at 75% masking.

Measure it instead of believing it. Time one encoder forward pass at each ratio — with a warm-up
call first, because the first call in the process pays for lazy initialisation and would otherwise
be the largest number in your table by a factor of ten — and plot time against ratio.

The measured speed-up is **much smaller than 16×**, and understanding why is the point of the task.
Three reasons, all real: at these sequence lengths the quadratic attention term is not what
dominates the runtime, the per-token feed-forward work is linear rather than quadratic, and there
is a fixed cost per call that does not shrink at all. The lecture's number is an asymptotic
statement about one term in the cost; your measurement is the whole system. Both are correct and
they answer different questions.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الحساب، مقيسًا

كان ادّعاء المحاضرة أن الإخفاء يُرخِص المُرمِّز **تربيعيًا**، لأن كلفة الانتباه تسير مع مربّع طول
المتتالية: `49²` = ٢٬٤٠١ مقابل `196²` = ٣٨٬٤١٦، أي أقلّ ستّ عشرة مرّة بالضبط عند إخفاء ٧٥٪.

قِس ذلك بدل أن تُصدّقه. خُذ زمن تمريرة أمامية واحدة للمُرمِّز عند كل نسبة — مع نداء تسخين أولًا، لأن
أول نداء في العملية يدفع ثمن التهيئة الكسولة فيصير لولا ذلك أكبر رقم في جدولك بعشرة أضعاف — وارسم
الزمن مقابل النسبة.

والتسريع المقيس **أصغر بكثير من ستّة عشر ضعفًا**، وفهم السبب هو مقصد المهمة. وثلاثة أسباب، كلها
حقيقية: عند هذه الأطوال ليس الحدّ التربيعي للانتباه هو المسيطر على زمن التشغيل، وعمل التغذية
الأمامية لكل رمز خطّي لا تربيعي، وثمّة كلفة ثابتة لكل نداء لا تصغر إطلاقًا. فرقم المحاضرة قولٌ
مقاربيّ عن حدٍّ واحد من الكلفة؛ وقياسك هو النظام كله. وكلاهما صحيح ويُجيبان عن سؤالين مختلفين.

</div>

In [ ]:
# TODO: Time the encoder at each ratio (after a warm-up), add the seconds to SWEEP, plot time against ratio, and print the measured speed-up next to the theoretical one.
# مهمة: قِس زمن المُرمِّز عند كل نسبة (بعد تسخين)، وأضف الثواني إلى `SWEEP`، وارسم الزمن مقابل النسبة، واطبع التسريع المقيس بجوار النظري.

### Task 2.6 — throw the decoder away

Set the masking ratio to **0.0** and run the encoder. Nothing is hidden, the sequence is 197 again,
and what comes back is a representation of the whole image.

That is the MAE as a feature extractor, and it is the only way anybody uses one after pretraining.
Take the `[CLS]` row for each of the 400 images and save it. Tomorrow those features are one row of
the comparison table.

Then the written task, one sentence: **what did you just throw away, and why was it never the
product?** The decoder is eight transformer blocks that exist to turn a partial encoding back into
pixels. Producing pixels was the pretext — the excuse to make the encoder learn something. Nobody
wanted the pixels; we already had the pixels. Keeping the decoder would be keeping the scaffolding
after the building is up.

Note while you are here how asymmetric the two halves are, and print the parameter counts. The
encoder is the large one, and it is the one that survives.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — ارمِ المُفكِّك

اضبط نسبة الإخفاء على **٠٫٠** وشغّل المُرمِّز. لا شيء مُخفى، وطول المتتالية ١٩٧ من جديد، والعائد تمثيل
للصورة كاملة.

وهذا هو المُرمِّز المُقنَّع مُستخرِجَ تمثيلات، وهي الطريقة الوحيدة التي يستعمله بها أحد بعد التدريب
المسبق. خذ صفّ `[CLS]` لكل صورة من الأربعمئة واحفظه. فتلك التمثيلات غدًا صفٌّ من صفوف جدول المقارنة.

ثم المهمة المكتوبة، جملة واحدة: **ماذا رميتَ للتوّ، ولماذا لم يكن منتَجًا قط؟** المُفكِّك ثماني كتل
محوّل موجودة لتُعيد ترميزًا جزئيًا إلى بكسلات. وإنتاج البكسلات كان الذريعة — الحجّة التي تجعل المُرمِّز
يتعلّم شيئًا. ولم يُرد أحد البكسلات؛ فالبكسلات عندنا أصلًا. والاحتفاظ بالمُفكِّك احتفاظٌ بالسقالة بعد
قيام البناء.

ولاحظ وأنت هنا كم النصفان غير متناظرين، واطبع عددَي المعاملات. فالمُرمِّز هو الكبير، وهو الذي يبقى.

</div>

In [ ]:
FEATURE_BATCH = 25

# TODO: Set the ratio to 0, extract the [CLS] feature for all 400 images in batches, and print the feature shape together with the encoder and decoder parameter counts.
# مهمة: اضبط النسبة على صفر، واستخرج تمثيل `[CLS]` لكل الصور الأربعمئة دفعاتٍ، واطبع شكل التمثيلات مع عددَي معاملات المُرمِّز والمُفكِّك.

**Your sentence.** Replace this line: what did you throw away, and why was it never the product?

<div dir="rtl" align="right">

**جملتك.** استبدل هذا السطر: ماذا رميت، ولماذا لم يكن منتَجًا قط؟

</div>

## Section 3 — Stretch: block masking  (≈30 min)

Random masking at 75% leaves a visible patch next to almost every hidden one. Take that away.

Instead of choosing the masked patches at random, mask a **contiguous block** covering the centre
of the image — the same 75%, arranged differently. You already have the mechanism: the model keeps
the patches with the lowest `noise`, so hand it a noise vector that increases towards the centre of
the 14×14 grid and the centre gets deleted.

Reconstruct at the same ratio and compare. The error is much higher — expect roughly double — and
the pictures show why: the model is now extrapolating into a hole rather than interpolating between
neighbours.

Then two sentences on what that says about the model. Random masking at 75% *sounds* brutal and is
in fact a local task, largely solvable by copying texture from a patch two positions away. Block
masking is the same ratio and a genuinely different task, because the answer is not adjacent to
anything the model can see. The gap between the two numbers is a measurement of how much of MAE's
apparent understanding is local interpolation — which is not a criticism of MAE, but it is the
reason nobody quotes a masking ratio without saying what the masking pattern was.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: إخفاء الكتل (نحو ٣٠ دقيقة)

الإخفاء العشوائي عند ٧٥٪ يترك رقعةً ظاهرة بجوار كل رقعة مُخفاة تقريبًا. انزع ذلك.

فبدل اختيار الرقع المُخفاة عشوائيًا، أخفِ **كتلة متّصلة** تُغطّي وسط الصورة — النسبة نفسها ٧٥٪
مرتّبةً ترتيبًا آخر. والآلية عندك أصلًا: يُبقي النموذج الرقع ذات `noise` الأدنى، فأعطِه متّجه ضوضاء
يتزايد نحو مركز الشبكة ١٤×١٤ فيُحذف المركز.

أعِد البناء عند النسبة نفسها وقارن. الخطأ أعلى بكثير — توقّع نحو الضعف — والصور تُظهر السبب: فالنموذج
الآن يستقرئ داخل ثقب لا يستوسط بين جيران.

ثم جملتان عمّا يقوله ذلك عن النموذج. فالإخفاء العشوائي عند ٧٥٪ **يبدو** قاسيًا وهو في الحقيقة مهمّة
محلّية، تُحلّ في معظمها بنسخ النسيج من رقعة على بُعد موضعين. وإخفاء الكتل النسبة نفسها ومهمّة مختلفة
حقًّا، لأن الجواب ليس مجاورًا لشيء يراه النموذج. والفجوة بين الرقمين قياسٌ لمقدار ما في فهم المُرمِّز
المُقنَّع الظاهري من استيساطٍ محلّي — وليس هذا نقدًا له، لكنه سبب ألّا يذكر أحد نسبة إخفاء دون أن يقول
ما كان نمط الإخفاء.

</div>

In [ ]:
# TODO: Build a centre-heavy noise vector, reconstruct at 0.75 with it, compare the masked MSE against random masking at the same ratio, and show the two side by side.
# مهمة: ابنِ متّجه ضوضاء مركزيّ الثقل، وأعِد البناء به عند ٠٫٧٥، وقارن خطأ الرقع المُخفاة بالإخفاء العشوائي عند النسبة نفسها، واعرض الاثنين جنبًا إلى جنب.

**Your two sentences.** Why is random masking easier than block masking at the same ratio, and what
does the gap tell you about what the model is doing?

<div dir="rtl" align="right">

**جملتاك.** لماذا الإخفاء العشوائي أسهل من إخفاء الكتل عند النسبة نفسها، وماذا تخبرك الفجوة عمّا
يفعله النموذج؟

</div>

## Save your artefact

`mae_sweep.parquet` — the four ratios, the visible token count, the masked-patch MSE and the
measured encoder time.

`mae_features.parquet` — the 768-dim `[CLS]` feature for all 400 images, extracted at
`mask_ratio=0.0`. **Tomorrow loads this file** as the third row of the comparison table, next to
your autoencoder codes from yesterday and DINOv2's features.

`mae_recon/` — the triple, the four ratio grids, the timing plot and the block-versus-random
comparison.

<div dir="rtl" align="right">

## احفظ أثرك

`mae_sweep.parquet` — النسب الأربع، وعدد الرموز الظاهرة، وخطأ الرقع المُخفاة، وزمن المُرمِّز المقيس.

`mae_features.parquet` — تمثيل `[CLS]` ذو الـ٧٦٨ بُعدًا لكل الصور الأربعمئة، مُستخرَجًا عند
`mask_ratio=0.0`. **ويُحمّل الغد هذا الملف** صفًّا ثالثًا في جدول المقارنة، بجوار شيفرات مُرمِّزك
الذاتي من أمس وتمثيلات DINOv2.

`mae_recon/` — الثلاثية، وشبكات النسب الأربع، ورسم الزمن، ومقارنة الكتلة بالعشوائي.

</div>

In [ ]:
SWEEP["block_masking_mse"] = [BLOCK_MSE if r == 0.75 else np.nan for r in SWEEP.ratio]
SWEEP_PATH = ARTEFACT_DIR / "mae_sweep.parquet"
SWEEP.to_parquet(SWEEP_PATH, index=False)

# One concat rather than 768 inserts — the column-at-a-time form is quadratic and pandas says so.
features_frame = pd.concat([
    pd.DataFrame({"file": [f"{p.parent.name}/{p.name}" for p in PATHS], "label": LABELS}),
    pd.DataFrame(MAE_FEATURES, columns=[f"f{d:03d}" for d in range(MAE_FEATURES.shape[1])]),
], axis=1)

FEATURES_PATH = ARTEFACT_DIR / "mae_features.parquet"
features_frame.to_parquet(FEATURES_PATH, index=False)

print(SWEEP.to_string(index=False))
print(f"\n{SWEEP_PATH.name}: {len(SWEEP)} rows")
print(f"{FEATURES_PATH.name}: {features_frame.shape[0]} x {MAE_FEATURES.shape[1]} features")
print(f"{len(list(RECON_DIR.glob('*.png')))} figures in {RECON_DIR.name}/")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(PARTITION_HOLDS,
      f"mask_patches must return a genuine partition at every ratio — the visible and masked "
      f"indices must be disjoint, cover all {N_PATCHES} patches, and have the right counts. "
      f"Dropping patches here corrupts every number downstream and raises nothing",
      f"يجب أن تُعيد `mask_patches` تجزئةً تامّة عند كل نسبة — فالفهارس الظاهرة والمُخفاة منفصلة، "
      f"وتُغطّي الرقع الـ{N_PATCHES} كلها، وأعدادها صحيحة. وإسقاط رقعٍ هنا يُفسد كل رقم لاحق ولا "
      f"يرفع شيئًا")

check(VISIBLE_PATCHES == 49 and ENCODER_TOKENS == 50,
      f"at mask_ratio 0.75 on a 196-patch image the encoder must process exactly 49 visible "
      f"patches plus [CLS] = 50 tokens — got {VISIBLE_PATCHES} and {ENCODER_TOKENS}",
      f"عند نسبة إخفاء ٠٫٧٥ على صورة ذات ١٩٦ رقعة يجب أن يعالج المُرمِّز ٤٩ رقعة ظاهرة بالضبط زائد "
      f"`[CLS]` = ٥٠ رمزًا — والناتج {VISIBLE_PATCHES} و{ENCODER_TOKENS}")

check(MSE_AGREES,
      f"your masked-only MSE must equal the model's own loss — got "
      f"{SWEEP.masked_mse.round(5).tolist()} against {SWEEP.model_loss.round(5).tolist()}. If they "
      f"differ, the mask is inverted and you are scoring the patches the model was given",
      f"يجب أن يساوي خطؤك على المُخفى وحده خسارةَ النموذج نفسه — والناتج "
      f"{SWEEP.masked_mse.round(5).tolist()} مقابل {SWEEP.model_loss.round(5).tolist()}. فإن "
      f"اختلفا فالقناع مقلوب وأنت تُقيّم الرقع التي أُعطيت للنموذج")

check(MSE_RISES,
      f"masked-patch MSE must increase with the masking ratio — got "
      f"{SWEEP.masked_mse.round(4).tolist()} at ratios {SWEEP.ratio.tolist()}",
      f"يجب أن يزيد خطأ الرقع المُخفاة مع نسبة الإخفاء — والناتج "
      f"{SWEEP.masked_mse.round(4).tolist()} عند النسب {SWEEP.ratio.tolist()}")

check(TIME_FALLS,
      f"the encoder's forward pass must get faster as the ratio rises, because it processes fewer "
      f"tokens — got {[round(s * 1000, 1) for s in SWEEP.encoder_seconds]} ms at ratios "
      f"{SWEEP.ratio.tolist()}. If it does not, you timed the decoder as well",
      f"يجب أن تُسرع تمريرة المُرمِّز كلّما ارتفعت النسبة لأنه يعالج رموزًا أقلّ — والناتج "
      f"{[round(s * 1000, 1) for s in SWEEP.encoder_seconds]} مللي ثانية عند النسب "
      f"{SWEEP.ratio.tolist()}. فإن لم يكن كذلك فقد قِستَ المُفكِّك أيضًا")

check(BLOCK_IS_HARDER,
      f"block masking at 0.75 must score strictly worse than random masking at 0.75 — got "
      f"{BLOCK_MSE:.4f} against {RANDOM_MSE:.4f}. Same ratio, different task",
      f"يجب أن يكون إخفاء الكتل عند ٠٫٧٥ أسوأ قطعًا من الإخفاء العشوائي عند ٠٫٧٥ — والناتج "
      f"{BLOCK_MSE:.4f} مقابل {RANDOM_MSE:.4f}. النسبة نفسها ومهمّة مختلفة")

check(MAE_FEATURES.shape == (len(PATHS), 768) and FULL_SEQUENCE == 197
      and not hasattr(features_frame, "decoder"),
      f"the saved features must be one 768-dim [CLS] vector per image, extracted at mask_ratio 0 "
      f"where the sequence is 197 — got {MAE_FEATURES.shape} and sequence {FULL_SEQUENCE}. The "
      f"decoder is not part of what gets saved",
      f"يجب أن تكون التمثيلات المحفوظة متّجه `[CLS]` واحدًا بـ٧٦٨ بُعدًا لكل صورة، مُستخرَجًا عند نسبة "
      f"إخفاء صفر حيث المتتالية ١٩٧ — والناتج {MAE_FEATURES.shape} والمتتالية {FULL_SEQUENCE}. "
      f"والمُفكِّك ليس جزءًا ممّا يُحفظ")

report()

## What's next

**W6D4 — the linear probe, and the week's reckoning.** Tomorrow the four representations meet in
one table: your autoencoder codes from Tuesday, the MAE features you just saved, DINOv2's frozen
features, and W4D5's supervised ResNet-18 — all scored by the same weak classifier on the **same
folds as W4D5**, which is the only reason the comparison means anything.

It is also the most transferable skill of the week. By tomorrow evening you can take any
representation, including one you built yourself, and say what it is worth in one number — and say
precisely what that number measures that a fine-tune does not.

Bring `ae_codes.parquet` and `mae_features.parquet`. Both are on disk. Assignment A6 is issued
tomorrow.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٦ اليوم ٤ — الفحص الخطيّ، وحساب الأسبوع.** غدًا تلتقي التمثيلات الأربعة في جدول واحد:
شيفرات مُرمِّزك الذاتي من الثلاثاء، وتمثيلات المُرمِّز المُقنَّع التي حفظتها للتوّ، وتمثيلات DINOv2
المجمّدة، وResNet-18 المُشرَف من الأسبوع الرابع — كلها مُقيَّمة بالمصنّف الضعيف نفسه على **التقسيمات
نفسها**، وهذا وحده سبب دلالة المقارنة على شيء.

وهي أيضًا أكثر مهارات الأسبوع قابليةً للنقل. فبمساء الغد تستطيع أن تأخذ أي تمثيل، بما فيه تمثيلٌ بنيته
بنفسك، وتقول كم يساوي في رقم واحد — وتقول بدقّة ما الذي يقيسه ذلك الرقم ولا يقيسه الضبط الدقيق.

أحضِر `ae_codes.parquet` و`mae_features.parquet`. وكلاهما على القرص. ويُسلَّم التكليف السادس غدًا.

</div>